# Download Music4All-Onion

Downloads the Music4All-Onion files we use from [Zenodo](https://zenodo.org/records/15394646) into `datasets/music4all-onion/`, decompressing each `.tsv.bz2` to `.tsv`. The dataset's `README.md` (feature descriptions) is downloaded too.

- Files that already exist are skipped, so it is safe to re-run.
- Partial downloads are resumed.
- Needs about **~15 GB** of free disk space once extracted (`userid_trackid_timestamp.tsv` alone is ~11 GB). Set `INCLUDE_TIMESTAMPS = False` to skip it.
- Music4All itself (audio, lyrics, metadata) must be requested manually. See the README.

In [ ]:
import bz2
import shutil
from pathlib import Path
from urllib.request import Request, urlopen

ZENODO_RECORD = "15394646"
BASE_URL = f"https://zenodo.org/api/records/{ZENODO_RECORD}/files"
INCLUDE_TIMESTAMPS = True  # ~2.2 GB download / ~11 GB extracted

FILES = [
    "userid_trackid_count",       # user-track play counts (collaborative filtering)
    "id_musicnn",                 # musicnn audio embeddings
    "id_mfcc_stats",              # MFCC statistics
    "id_essentia",                # Essentia audio descriptors
    "id_genres_tf-idf",           # genre tf-idf
    "id_tags_dict",               # Last.fm tags with weights (sparse; the tf-idf version is ~31 GB extracted)
]
if INCLUDE_TIMESTAMPS:
    FILES.append("userid_trackid_timestamp")  # timestamped listens (sequence models)

# Run from the repo root or from the notebook's folder
ROOT = Path.cwd()
if not (ROOT / "datasets").exists() and (ROOT.parent / "datasets").exists():
    ROOT = ROOT.parent
DEST = ROOT / "datasets" / "music4all-onion"
DEST.mkdir(parents=True, exist_ok=True)
print("Destination:", DEST)

In [ ]:
CHUNK = 1024 * 1024

def download(name: str) -> Path:
    archive = DEST / f"{name}.tsv.bz2"
    url = f"{BASE_URL}/{name}.tsv.bz2/content"
    done = archive.stat().st_size if archive.exists() else 0

    # Ask for the total size, then resume from what we already have
    with urlopen(Request(url, method="HEAD")) as r:
        total = int(r.headers["Content-Length"])
    if done == total:
        return archive
    headers = {"Range": f"bytes={done}-"} if 0 < done < total else {}
    mode = "ab" if headers else "wb"
    with urlopen(Request(url, headers=headers)) as r, open(archive, mode) as out:
        got = done if headers else 0
        while chunk := r.read(CHUNK):
            out.write(chunk)
            got += len(chunk)
            print(f"\r  {name}: {got / total:6.1%} of {total / 1e6:,.0f} MB", end="")
    print()
    return archive

def extract(archive: Path) -> None:
    tsv = archive.with_suffix("")  # strips .bz2
    tmp = tsv.with_suffix(".tsv.part")
    with bz2.open(archive, "rb") as src, open(tmp, "wb") as dst:
        shutil.copyfileobj(src, dst, CHUNK)
    tmp.rename(tsv)  # only appears once fully extracted
    archive.unlink()

def download_readme() -> None:
    readme = DEST / "README.md"
    if readme.exists():
        print("README.md already exists, skipping")
        return
    with urlopen(f"{BASE_URL}/README.md/content") as r:
        readme.write_bytes(r.read())
    print("Downloaded README.md")

download_readme()

for name in FILES:
    tsv = DEST / f"{name}.tsv"
    if tsv.exists():
        print(f"{name}.tsv already exists, skipping")
        continue
    print(f"Downloading {name} ...")
    archive = download(name)
    print("  extracting ...")
    extract(archive)
    print(f"  done: {tsv.name}")

In [ ]:
for p in sorted(DEST.glob("*.tsv")):
    print(f"{p.name:35s} {p.stat().st_size / 1e6:>10,.0f} MB")